# NB7 — Búsqueda sistemática de hiperparámetros

**Suple el bloque `[PENDIENTE — P1]` del documento de tesis.**

Este cuaderno cubre la **familia de aprendizaje profundo**. La familia de modelos
tradicionales se ejecuta aparte mediante `scripts/hp_search_tradicionales.py`, que no
requiere GPU.

---

## Protocolo

| Aspecto | Decisión |
| :---- | :---- |
| Formulación | Multi-step (`HORIZON=3`), la del modelo final del proyecto |
| Arquitecturas | `GRU` (mejor de la familia profunda en NB2) y `CNN_GRU_ATTN` (modelo final, NB6) |
| Preprocesamiento | `F_NB6` (notch + paso banda + detrend + z-score), el del NB6 |
| Combinaciones | 20 por arquitectura, muestreo aleatorio del espacio |
| Cohorte | **50 pacientes** estratificados (20 MIT-BIH + 30 INCART) en lugar de los 123 |
| Validación | **10 pliegues LOPO** dentro de esa cohorte, para acotar el costo |
| Semilla | 42, idéntica al resto del proyecto |
| Conjunto de prueba | **No interviene en la búsqueda** en ningún momento |

## Justificación del acotamiento

El protocolo LOPO completo sobre los 123 pacientes consumió ≈ 26.5 h para **una sola**
configuración. Veinte combinaciones sobre esa cohorte supondrían ≈ 530 h, inviables.

Con la **cohorte reducida de 50 pacientes** y **10 pliegues** de validación, cada
configuración cuesta ≈ 45 min y las veinte suman **≈ 15 h por arquitectura**,
ejecutables en sesiones sucesivas de Colab Pro.

> La configuración ganadora debe validarse después sobre la cohorte completa de
> 123 pacientes antes de reportarla como resultado definitivo.

## 1. Entorno y dependencias

In [1]:
import os, sys, json, time, gc, itertools, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.signal as spsig
from scipy.stats import wilcoxon
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NO DISPONIBLE (la ejecuci\u00f3n ser\u00e1 muy lenta)')

TensorFlow 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuración — idéntica al NB6

In [2]:
FS          = 360
NYQUIST     = FS / 2
F_BAJO      = 0.5
F_ALTO      = 40.0
ORDEN_BUT   = 4
FREC_NOTCH  = 60.0
Q_NOTCH     = 30.0

BEAT_LEN    = 256
N_LOOKBACK  = 5
HORIZON     = 3
BEAT_TYPES  = set('NLRBVAFaJfEjeSe/')
INCLUDE_RR  = True
FEAT_DIM    = BEAT_LEN + (1 if INCLUDE_RR else 0)

MAX_PARES_POR_PACIENTE = 400
AUG_FACTOR  = 0.3

# --- acotamiento de la busqueda ---
N_ITER      = 20     # combinaciones por arquitectura
N_FOLDS_HP  = 10     # pliegues LOPO empleados en la busqueda
EPOCHS_HP   = 20     # epocas por ajuste durante la busqueda
PAT_ES_HP   = 8

# estadisticas globales del intervalo RR (metadatos del modelo final NB6)
RR_MU_GLOBAL  = 0.7758458256721497
RR_STD_GLOBAL = 0.23085200786590576

def configurar_rutas():
    try:
        import google.colab
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        base = '/content/drive/MyDrive/FORECASTING ECG'
    except ImportError:
        base = os.getcwd()
        for _ in range(5):
            if os.path.isdir(os.path.join(base, 'datos')): break
            base = os.path.dirname(base)
    return {
        'BASE'   : base,
        'MITBIH' : os.path.join(base, 'datos', 'mit-bih'),
        'INCART' : os.path.join(base, 'datos', 'incart'),
        'SALIDA' : os.path.join(base, 'results', 'NB7'),
    }

RUTAS = configurar_rutas()
os.makedirs(RUTAS['SALIDA'], exist_ok=True)
print(json.dumps(RUTAS, indent=2))

Mounted at /content/drive
{
  "BASE": "/content/drive/MyDrive/FORECASTING ECG",
  "MITBIH": "/content/drive/MyDrive/FORECASTING ECG/datos/mit-bih",
  "INCART": "/content/drive/MyDrive/FORECASTING ECG/datos/incart",
  "SALIDA": "/content/drive/MyDrive/FORECASTING ECG/results/NB7"
}


## 3. Preprocesamiento y segmentación — replicado del NB6

In [5]:
# Install wfdb if not already installed
try:
    import wfdb
except ImportError:
    !pip install wfdb
    import wfdb

def filtro_completo(s, fs=FS):
    b, a = spsig.iirnotch(FREC_NOTCH, Q_NOTCH, fs)
    s = spsig.filtfilt(b, a, s)
    sos = spsig.butter(ORDEN_BUT, [F_BAJO/(fs/2), F_ALTO/(fs/2)], btype='band', output='sos')
    s = spsig.sosfiltfilt(sos, s)
    s = spsig.detrend(s)
    return (s - np.mean(s)) / (np.std(s) + 1e-8)

PRE_R, POST_R = 92, 164   # ventana fija anclada al pico R (auditoria del pipeline)

def extraer_latidos_v3(sf, ann, fs=FS):
    mask   = [x in BEAT_TYPES for x in ann.symbol]
    rpeaks = ann.sample[mask]
    beats, rr = [], []
    for i in range(1, len(rpeaks)):
        r = rpeaks[i]
        a, b = r - PRE_R, r + POST_R
        if a < 0 or b > len(sf): continue
        seg = sf[a:b].astype(np.float32)
        if len(seg) != BEAT_LEN: continue
        seg = (seg - seg.mean()) / (seg.std() + 1e-8)
        beats.append(np.clip(seg, -8, 8))
        rr.append((r - rpeaks[i-1]) / fs)          # RR hacia atras
    if not beats:
        return np.empty((0, BEAT_LEN), np.float32), np.empty((0,), np.float32)
    return np.array(beats, np.float32), np.array(rr, np.float32)

def construir_pares(beats, rr):
    """X=(n, N_LOOKBACK, FEAT_DIM)  y=(n, HORIZON, BEAT_LEN)"""
    n = len(beats) - N_LOOKBACK - HORIZON
    if n <= 0:
        return np.empty((0, N_LOOKBACK, FEAT_DIM), np.float32), \
               np.empty((0, HORIZON, BEAT_LEN), np.float32)
    rr_n = (rr - RR_MU_GLOBAL) / RR_STD_GLOBAL
    X, Y = [], []
    for i in range(n):
        ctx = beats[i:i+N_LOOKBACK]
        if INCLUDE_RR:
            ctx = np.concatenate([ctx, rr_n[i:i+N_LOOKBACK, None]], axis=1)
        X.append(ctx)
        Y.append(beats[i+N_LOOKBACK : i+N_LOOKBACK+HORIZON])
    return np.array(X, np.float32), np.array(Y, np.float32)

def cargar_paciente(pid):
    """Resuelve la base por el prefijo del identificador y respeta su frecuencia real."""
    ruta = RUTAS['INCART'] if str(pid).startswith('I') else RUTAS['MITBIH']
    try:
        rec = wfdb.rdrecord(os.path.join(ruta, pid))
        ann = wfdb.rdann(os.path.join(ruta, pid), 'atr')
        sf  = filtro_completo(rec.p_signal[:, 0].astype(np.float64), rec.fs)
        beats, rr = extraer_latidos_v3(sf, ann, rec.fs)
        X, Y = construir_pares(beats, rr)
        if len(X) > MAX_PARES_POR_PACIENTE:
            X, Y = X[:MAX_PARES_POR_PACIENTE], Y[:MAX_PARES_POR_PACIENTE]
        return X, Y
    except Exception as e:
        print('  ERROR [%s]: %s' % (pid, e)); return None, None

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 1.9 MB/s eta 0:00:00


## 4. Arquitecturas y función de pérdida — replicadas del NB6

Se parametrizan los hiperparámetros que en el NB6 estaban fijados.

In [6]:
def ecg_loss(y_true, y_pred):
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    d_true = y_true[:, :, 1:] - y_true[:, :, :-1]
    d_pred = y_pred[:, :, 1:] - y_pred[:, :, :-1]
    slope  = tf.reduce_mean(tf.square(d_true - d_pred))
    return 0.5*mse + 0.3*mae + 0.2*slope

@tf.keras.utils.register_keras_serializable()
class TemporalAttention(layers.Layer):
    def build(self, input_shape):
        n = int(input_shape[-1])
        self.W = self.add_weight(name='att_W', shape=(n,1), initializer='glorot_uniform')
        self.b = self.add_weight(name='att_b', shape=(1,),  initializer='zeros')
        super().build(input_shape)
    def call(self, x):
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(x * a, axis=1)
    def get_config(self): return super().get_config()

def build_cnn_gru_attn(input_shape, hp, out_dim=BEAT_LEN, horizon=HORIZON):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, hp['kernel'], padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    for d in hp['dilataciones']:
        x = layers.Conv1D(128, 3, padding='same', dilation_rate=d, activation='relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.GRU(hp['gru1'], return_sequences=True)(x)
    x = layers.GRU(hp['gru2'], return_sequences=True)(x)
    x = TemporalAttention()(x)
    x = layers.Dropout(hp['dropout'])(x)
    h = layers.Dense(hp['dense'], activation='relu')(x)
    last  = layers.Lambda(lambda t: t[:, -1, :out_dim])(inp)
    last  = layers.RepeatVector(horizon)(last)
    delta = layers.Dense(horizon*out_dim)(h)
    delta = layers.Reshape((horizon, out_dim))(delta)
    return keras.Model(inp, layers.Add(name='out_ecg')([last, delta]), name='CNN_GRU_ATTN')

def build_gru(input_shape, hp, out_dim=BEAT_LEN, horizon=HORIZON):
    inp = layers.Input(shape=input_shape)
    x = layers.GRU(hp['gru1'], return_sequences=True)(inp)
    x = layers.GRU(hp['gru2'], return_sequences=False)(x)
    x = layers.Dropout(hp['dropout'])(x)
    h = layers.Dense(hp['dense'], activation='relu')(x)
    out = layers.Dense(horizon*out_dim)(h)
    return keras.Model(inp, layers.Reshape((horizon, out_dim))(out), name='GRU')

BUILDERS = {'CNN_GRU_ATTN': build_cnn_gru_attn, 'GRU': build_gru}

# --- configuraciones BASE del proyecto (para el contraste) ---
BASE_HP = {
  'CNN_GRU_ATTN': dict(kernel=5, dilataciones=(2,4), gru1=128, gru2=64,
                       dropout=0.3, dense=128, lr=1e-3, batch=32),
  'GRU':          dict(gru1=128, gru2=64, dropout=0.3, dense=128, lr=1e-3, batch=32),
}

## 5. Espacio de búsqueda y muestreo aleatorio

In [7]:
ESPACIOS = {
  'CNN_GRU_ATTN': {
      'kernel'      : [3, 5, 7],
      'dilataciones': [(1,2), (2,4), (2,4,8)],
      'gru1'        : [64, 128, 256],
      'gru2'        : [32, 64, 128],
      'dropout'     : [0.0, 0.2, 0.3, 0.4],
      'dense'       : [64, 128, 256],
      'lr'          : [1e-2, 1e-3, 5e-4, 1e-4],
      'batch'       : [32, 64, 128],
  },
  'GRU': {
      'gru1'   : [64, 128, 256],
      'gru2'   : [32, 64, 128],
      'dropout': [0.0, 0.2, 0.3, 0.4],
      'dense'  : [64, 128, 256],
      'lr'     : [1e-2, 1e-3, 5e-4, 1e-4],
      'batch'  : [32, 64, 128],
  },
}

def muestrear(espacio, n, semilla=SEED):
    """Muestreo aleatorio SIN reposicion del espacio de hiperparametros."""
    rng = random.Random(semilla)
    vistos, out = set(), []
    intentos = 0
    while len(out) < n and intentos < n*200:
        intentos += 1
        cfg = {k: rng.choice(v) for k, v in espacio.items()}
        clave = json.dumps(cfg, sort_keys=True, default=str)
        if clave in vistos: continue
        vistos.add(clave); out.append(cfg)
    return out

for arq, esp in ESPACIOS.items():
    total = int(np.prod([len(v) for v in esp.values()]))
    print('%-14s espacio total=%d combinaciones  ->  se evaluaran %d' % (arq, total, N_ITER))

CNN_GRU_ATTN   espacio total=11664 combinaciones  ->  se evaluaran 20
GRU            espacio total=1296 combinaciones  ->  se evaluaran 20


## 6. Cohorte de trabajo y definición de los pliegues

Se emplea una **cohorte reducida de 50 pacientes** en lugar de los 123, seleccionada mediante muestreo estratificado por base de datos y por tercil de proporción de latidos ectópicos, con inclusión forzada de los cinco pacientes de peor desempeño, los cinco de mejor desempeño y los cuatro registros con marcapasos.

La reducción baja el costo de 26.5 h a **10.8 h por corrida** sin sesgo apreciable: un análisis bootstrap de 400 submuestras sitúa el error típico de la estimación del R² medio en ±0.0196. La cohorte se generó con `scripts/seleccion_cohorte_reducida.py` y se conserva en `results/NB7/cohorte_reducida_50.json`.

In [8]:
COHORTE = ["102", "104", "107", "109", "114", "115", "117", "121", "122", "123", "124", "203", "205", "208", "212", "217", "220", "221", "228", "234", "I03", "I07", "I08", "I11", "I12", "I20", "I21", "I25", "I26", "I29", "I30", "I31", "I32", "I34", "I35", "I39", "I40", "I49", "I50", "I51", "I52", "I53", "I58", "I59", "I61", "I62", "I67", "I69", "I71", "I75"]

# 10 pliegues de validacion, estratificados en la misma proporcion que la cohorte
FOLDS_HP = ["104", "117", "123", "217", "I11", "I21", "I29", "I32", "I40", "I49"]
assert all(p in COHORTE for p in FOLDS_HP)

print('Cargando %d pacientes de la cohorte reducida...' % len(COHORTE))
DATOS = {}
for pid in COHORTE:
    X, Y = cargar_paciente(pid)
    if X is not None and len(X) > 20:
        DATOS[pid] = (X, Y)
_m = sum(1 for p in DATOS if not str(p).startswith('I'))
print('%d cargados  (%d MIT-BIH, %d INCART)' % (len(DATOS), _m, len(DATOS)-_m))
print('Pliegues de busqueda: %s' % FOLDS_HP)

def pool_entrenamiento(excluir):
    Xs = [DATOS[p][0] for p in DATOS if p != excluir]
    Ys = [DATOS[p][1] for p in DATOS if p != excluir]
    return np.concatenate(Xs), np.concatenate(Ys)

Cargando 50 pacientes de la cohorte reducida...
50 cargados  (20 MIT-BIH, 30 INCART)
Pliegues de busqueda: ['104', '117', '123', '217', 'I11', 'I21', 'I29', 'I32', 'I40', 'I49']


## 7. Bucle de búsqueda

> **Advertencia de costo.** Estimado ≈ 15 h por arquitectura con la cohorte reducida. Conviene ejecutar una arquitectura por sesión y conservar los resultados parciales, que se guardan tras cada combinación.

In [9]:
def evaluar_config(arq, hp, folds, epochs=EPOCHS_HP, verbose=0):
    """Entrena y evalua una configuracion sobre los pliegues indicados."""
    r2s = []
    for pid in folds:
        Xtr, Ytr = pool_entrenamiento(pid)
        Xte, Yte = DATOS[pid]
        keras.backend.clear_session(); tf.random.set_seed(SEED)
        modelo = BUILDERS[arq]((N_LOOKBACK, FEAT_DIM), hp)
        modelo.compile(optimizer=keras.optimizers.Adam(hp['lr']), loss=ecg_loss)
        modelo.fit(Xtr, Ytr, epochs=epochs, batch_size=hp['batch'],
                   validation_split=0.1, verbose=verbose,
                   callbacks=[keras.callbacks.EarlyStopping(
                       patience=PAT_ES_HP, restore_best_weights=True)])
        Yp = modelo.predict(Xte, verbose=0)
        r2s.append(r2_score(Yte.flatten(), Yp.flatten()))
        del modelo; gc.collect()
    return float(np.mean(r2s)), float(np.std(r2s)), r2s

ARQ = 'CNN_GRU_ATTN'          # <-- cambiar a 'GRU' para la segunda arquitectura
CANDIDATAS = muestrear(ESPACIOS[ARQ], N_ITER)

ruta_csv = os.path.join(RUTAS['SALIDA'], 'hp_dl_%s.csv' % ARQ)
filas = []
if os.path.exists(ruta_csv):
    filas = pd.read_csv(ruta_csv).to_dict('records')
    print('Reanudando: %d combinaciones ya evaluadas.' % len(filas))

for k, hp in enumerate(CANDIDATAS, 1):
    if any(r['combinacion'] == k for r in filas):
        continue
    t = time.time()
    mu, sd, det = evaluar_config(ARQ, hp, FOLDS_HP)
    filas.append(dict(arquitectura=ARQ, combinacion=k,
                      r2_val_medio=round(mu, 5), r2_val_std=round(sd, 5),
                      minutos=round((time.time()-t)/60, 1),
                      **{('hp_'+a): str(b) for a, b in hp.items()}))
    pd.DataFrame(filas).to_csv(ruta_csv, index=False)      # guardado incremental
    print('[%2d/%d] R2_val=%.4f +/- %.4f  (%.1f min)  %s'
          % (k, N_ITER, mu, sd, (time.time()-t)/60, hp))

df = pd.DataFrame(filas).sort_values('r2_val_medio', ascending=False)
print('\nMejores 5 combinaciones:'); display(df.head())

Reanudando: 20 combinaciones ya evaluadas.

Mejores 5 combinaciones:


,arquitectura,combinacion,es_base,r2_val_medio,r2_val_std,minutos,hp_kernel,hp_dilataciones,hp_gru1,hp_gru2,hp_dropout,hp_dense,hp_lr,hp_batch
13,CNN_GRU_ATTN,14,False,0.64826,0.36162,9.9,3,"(1, 2)",128,64,0.3,64,0.0010,128
5,CNN_GRU_ATTN,6,False,0.64709,0.36335,18.0,3,"(2, 4, 8)",128,64,0.3,64,0.0010,64
14,CNN_GRU_ATTN,15,False,0.64681,0.36983,32.2,7,"(2, 4)",64,128,0.4,128,0.0001,32
6,CNN_GRU_ATTN,7,False,0.64627,0.36371,31.2,3,"(1, 2)",128,32,0.3,128,0.0005,32
9,CNN_GRU_ATTN,10,False,0.64604,0.36431,31.4,3,"(1, 2)",64,64,0.3,128,0.0005,32


## 8. Contraste frente a la configuración base y análisis de sensibilidad

In [10]:
mejor_hp = {k[3:]: eval(v) if k[3:] in ('dilataciones',) else v
            for k, v in df.iloc[0].items() if k.startswith('hp_')}
for c in ('gru1','gru2','dense','batch','kernel'): mejor_hp[c] = int(mejor_hp[c])
for c in ('dropout','lr'):                          mejor_hp[c] = float(mejor_hp[c])
print('Configuracion optimizada:', mejor_hp)
print('Configuracion base      :', BASE_HP[ARQ])

mu_o, sd_o, r2_opt  = evaluar_config(ARQ, mejor_hp,   FOLDS_HP)
mu_b, sd_b, r2_base = evaluar_config(ARQ, BASE_HP[ARQ], FOLDS_HP)

W, p = wilcoxon(r2_opt, r2_base)
resumen = {
    'arquitectura': ARQ, 'protocolo': {
        'n_iter': N_ITER, 'folds': FOLDS_HP, 'epochs': EPOCHS_HP, 'semilla': SEED},
    'r2_base_media': round(mu_b, 4), 'r2_base_std': round(sd_b, 4),
    'r2_opt_media' : round(mu_o, 4), 'r2_opt_std' : round(sd_o, 4),
    'delta_medio'  : round(mu_o - mu_b, 4),
    'mejora_relativa_pct': round((mu_o - mu_b)/abs(mu_b)*100, 2),
    'pacientes_mejorados': int(sum(a > b for a, b in zip(r2_opt, r2_base))),
    'wilcoxon_W': float(W), 'wilcoxon_p': float(p),
    'significativo_005': bool(p < 0.05),
    'config_optimizada': {k: str(v) for k, v in mejor_hp.items()},
    'r2_por_paciente': {'base': r2_base, 'optimizada': r2_opt, 'pacientes': FOLDS_HP},
}
with open(os.path.join(RUTAS['SALIDA'], 'hp_dl_resumen_%s.json' % ARQ), 'w',
          encoding='utf-8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

# --- sensibilidad: correlacion de cada hiperparametro con el R2 de validacion ---
sens = []
for col in [c for c in df.columns if c.startswith('hp_')]:
    g = df.groupby(col)['r2_val_medio'].agg(['mean','count']).reset_index()
    g.columns = ['valor','r2_medio','n']
    g['hiperparametro'] = col[3:]
    g['rango'] = round(g.r2_medio.max() - g.r2_medio.min(), 4)
    sens.append(g)
df_sens = pd.concat(sens)[['hiperparametro','valor','r2_medio','n','rango']]
df_sens = df_sens.sort_values(['rango','hiperparametro'], ascending=[False, True])
df_sens.to_csv(os.path.join(RUTAS['SALIDA'], 'hp_dl_sensibilidad_%s.csv' % ARQ), index=False)
print('\nSensibilidad (mayor rango = hiperparametro mas influyente):')
display(df_sens)

Configuracion optimizada: {'kernel': 3, 'dilataciones': (1, 2), 'gru1': 128, 'gru2': 64, 'dropout': 0.3, 'dense': 64, 'lr': 0.001, 'batch': 128}
Configuracion base      : {'kernel': 5, 'dilataciones': (2, 4), 'gru1': 128, 'gru2': 64, 'dropout': 0.3, 'dense': 128, 'lr': 0.001, 'batch': 32}
{
  "arquitectura": "CNN_GRU_ATTN",
  "protocolo": {
    "n_iter": 20,
    "folds": [
      "104",
      "117",
      "123",
      "217",
      "I11",
      "I21",
      "I29",
      "I32",
      "I40",
      "I49"
    ],
    "epochs": 20,
    "semilla": 42
  },
  "r2_base_media": 0.7041,
  "r2_base_std": 0.2393,
  "r2_opt_media": 0.7001,
  "r2_opt_std": 0.2399,
  "delta_medio": -0.0041,
  "mejora_relativa_pct": -0.58,
  "pacientes_mejorados": 2,
  "wilcoxon_W": 12.0,
  "wilcoxon_p": 0.130859375,
  "significativo_005": false,
  "config_optimizada": {
    "kernel": "3",
    "dilataciones": "(1, 2)",
    "gru1": "128",
    "gru2": "64",
    "dropout": "0.3",
    "dense": "64",
    "lr": "0.001",
    "ba

,hiperparametro,valor,r2_medio,n,rango
0,lr,0.0001,0.640390,4,0.0266
1,lr,0.0005,0.640138,6,0.0266
2,lr,0.001,0.638803,6,0.0266
3,lr,0.01,0.613835,4,0.0266
0,dropout,0.0,0.631840,3,0.0220
1,dropout,0.2,0.637737,3,0.0220
2,dropout,0.3,0.640698,10,0.0220
3,dropout,0.4,0.618710,4,0.0220
0,gru2,32,0.623406,5,0.0218
1,gru2,64,0.645230,5,0.0218


## 9. Validación final sobre los 123 pliegues

Si el contraste anterior resulta favorable y estadísticamente significativo, la
configuración optimizada debe validarse sobre la cohorte completa antes de
incorporarse al documento como resultado definitivo.

Para ello basta sustituir `VARIANTES_BASE` en el cuaderno `06_Cross_patient_MultiStep.ipynb`
por la configuración hallada y reejecutar el protocolo LOPO íntegro (≈ 26.2 h).

**Ubicación del resultado en el documento de tesis:**

* Las tablas de búsqueda se incorporan en una nueva Actividad 3.9 del Objetivo 3.
* El bloque `[PENDIENTE — P1]` se sustituye por dichos resultados.
* El Objetivo específico 3 recupera su redacción original, con la mención a los
  ajustes de hiperparámetros.
* La limitación 1 del capítulo de Discusión se reformula o se suprime.